# UdaciSense: Optimized Object Recognition
## Notebook 2: Compression Techniques

 
In this notebook, you'll explore different model compression techniques to meet the requirements:
- The optimized model should be **30% smaller** than the baseline
- The optimized model should **reduce inference time by 40%**
- The optimized model should **maintain accuracy within 5%** of the baseline

You can experiment with different methods:
1. **Post-training**: Quantization, pruning, graph optimizations.
2. **In-training**: Quantization, pruning, distillation.
Optionally, you can choose to implement other techniques too.

Make sure to experiment with at least two different techniques. 
You will need to combine the selected techniques into a multi-step compression pipeline next, so make sure to select techniques that seem  promising individually but also combined.

## Post-Training Quantization

Post-training quantization reduces model precision from floating point to lower precision representations such as INT8. This reduces memory footprint and can improve CPU inference latency, which is especially important for mobile devices.

This technique is appropriate for UdaciSense because the target environment includes mid-range and budget-friendly smartphones where memory bandwidth and CPU efficiency directly affect battery life and user experience.

### Step 1: Set up the environment

In [1]:
# Make sure that libraries are dynamically re-loaded if changed
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

In [2]:
# Import necessary libraries
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pprint
import random
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from compression.post_training.pruning import prune_model
from compression.post_training.quantization import quantize_model
from compression.post_training.graph_optimization import optimize_model, verify_model_equivalence
from compression.in_training.distillation import train_with_distillation, MobileNetV3_Household_Small
from compression.in_training.pruning import train_with_pruning
from compression.in_training.quantization import train_model_qat, QuantizableMobileNetV3_Household

from utils import MAX_ALLOWED_ACCURACY_DROP, TARGET_INFERENCE_SPEEDUP,TARGET_MODEL_COMPRESSION 
from utils.data_loader import get_household_loaders, get_input_size, print_dataloader_stats, visualize_batch
from utils.model import MobileNetV3_Household, load_model, save_model, print_model_summary
from utils.visualization import plot_multiple_models_comparison
from utils.compression import (
    compare_experiments, compare_optimized_model_to_baseline, evaluate_optimized_model, list_experiments,  # For experimentation
    is_quantized  # For quantization
)

In [3]:
# Ignore PyTorch deprecation warnings
import warnings
warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)
warnings.filterwarnings("ignore", category=UserWarning)  # Optional: Ignore all user warnings

In [ ]:
# Check if CUDA is available
devices = ["cpu"]
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    devices.extend([f"cuda:{i} ({torch.cuda.get_device_name(i)})" for i in range(num_devices)])
print(f"Devices available: {devices}")

In [5]:
# Create directories for each technique
compression_types = [
    "post_training/pruning",
    "post_training/quantization",
    "post_training/graph_optimization",
    "in_training/distillation", 
    "in_training/quantization",
    "in_training/pruning",
]
for comp_type in compression_types:
    models_dir = f"../models/{comp_type}"
    models_ckp_dir = f"{models_dir}/checkpoints"
    results_dir = f"../results/{comp_type}"
    
    os.makedirs(models_ckp_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

In [ ]:
# Set random seed for reproducibility
def set_deterministic_mode(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    
    return seed_worker

set_deterministic_mode(42)
g = torch.Generator()
g.manual_seed(42)

### Step 2: Load the dataset

In [ ]:
# Load household objects dataset
train_loader, test_loader = get_household_loaders(
    image_size="CIFAR", batch_size=128, num_workers=2,
)

# Get input_size
input_size = get_input_size("CIFAR")
print(f"Input has size: {input_size}")

# Get class names
class_names = train_loader.dataset.classes
print(f"Datasets have these classes: ")
for i in range(len(class_names)):
    print(f"  {i}: {class_names[i]}")

# Visualize some examples
for dataset_type, data_loader in [('train', train_loader), ('test', test_loader)]:
    print(f"\nInformation on {dataset_type} set")
    print_dataloader_stats(data_loader, dataset_type)
    print(f"Examples of images from the {dataset_type} set")
    visualize_batch(data_loader, num_images=10)

### Step 3: Load the baseline model and metrics

In [ ]:
# Load the baseline model
baseline_model = MobileNetV3_Household()
baseline_model_name = "baseline_mobilenet"
baseline_model.load_state_dict(torch.load(f"../models/{baseline_model_name}/checkpoints/model.pth"))
print_model_summary(baseline_model)

# Load baseline metrics
with open(f"../results/{baseline_model_name}/metrics.json", "r") as f:
    baseline_metrics = json.load(f)

print("\nBaseline Model Metrics:")
pprint.pp(baseline_metrics)

# Calculate target metrics based on CTO requirements
target_model_size = baseline_metrics['size']['model_size_mb'] * (1 - TARGET_MODEL_COMPRESSION)
target_inference_time_cpu = baseline_metrics['timing']['cpu']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
if torch.cuda.is_available():
    target_inference_time_gpu = baseline_metrics['timing']['cuda']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
min_acceptable_accuracy = baseline_metrics['accuracy']['top1_acc'] * (1 - MAX_ALLOWED_ACCURACY_DROP) 

print("Optimization Targets:")
print(f"Target Model Size: {baseline_metrics['size']['model_size_mb']:.2f} --> {target_model_size:.2f} MB ({TARGET_MODEL_COMPRESSION*100}% reduction)")
print(f"Target Inference Time (CPU): {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} --> {target_inference_time_cpu:.2f} ms ({TARGET_INFERENCE_SPEEDUP*100}% reduction)")
if torch.cuda.is_available():
    print(f"Target Inference Time (GPU): {baseline_metrics['timing']['cuda']['avg_time_ms']:.2f} --> {target_inference_time_gpu:.2f} ms ({TARGET_INFERENCE_SPEEDUP*100}% reduction)")
print(f"Minimum Acceptable Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f} --> {min_acceptable_accuracy:.2f} (within {MAX_ALLOWED_ACCURACY_DROP*100}% of baseline)")

### Step 4: Implement and evaluate compression techniques

Now you'll implement and evaluate different compression techniques. For each technique that you choose, you'll:
1. Implement the technique
2. Evaluate its impact on model size, inference time, and accuracy
3. Analyze the trade-offs

To choose a technique, simply uncomment the apply_{TECHNIQUE_NAME}_technique() function call in the corresponding technique cell block.

#### 4.1 Post-Training - Quantization

Quantization reduces the precision of weights and activations, converting floating-point values to integers.


In [ ]:
# Define a function to apply quantization and evaluate results
def apply_post_training_quantization(quantization_type, backend, device):
    """
    Apply quantization to a model with given method and backend.
    
    Args:
        quantization_type: Quantization method ("static" or "dynamic")
        backend: Backend for quantization ("fbgemm" for x86 or "qnnpack" for ARM)
        device: Which device to use for model loading, training, and evaluation
        
    Returns:
        Tuple of (optimized_model, comparison_results, experiment_name)
    """
    # Define unique experiment name given main parameters
    experiment_name = f"post_training/quantization/{quantization_type}"
    
    # Create experiment subdirectories
    os.makedirs(f"../models/{experiment_name}", exist_ok=True)
    os.makedirs(f"../results/{experiment_name}", exist_ok=True)
    
    print(f"Applying {quantization_type} quantization with {backend} backend")
    
    # Make a copy of the baseline model and move to specified device
    orig_model = load_model(f"../models/{baseline_model_name}/checkpoints/model.pth").to(device)
    orig_model.eval()
    
    # Apply post-training quantization
    # TODO: IMPLEMENT THIS FUNCTION IN THE compression/ FOLDER    
    quantized_model = quantize_model(
        orig_model,
        quantization_type=quantization_type,
        calibration_data_loader=train_loader if quantization_type == "static" else None,
        calibration_num_batches=1 if quantization_type == "static" else None,  # Set this to the desired value
        backend=backend,
    )
    
    # Save the quantized model
    save_model(quantized_model, f"../models/{experiment_name}/model.pth")
    
    # Check that model is indeed quantized
    is_quantized(quantized_model)
    
    # Evaluate quantized model
    evaluate_optimized_model(
        quantized_model, test_loader, experiment_name, class_names, input_size, device=torch.device('cpu')
    )
    
    # Compare with baseline model for performance differences
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        quantized_model,
        experiment_name,
        test_loader,
        class_names,
        device=torch.device('cpu'),
    )
    
    return quantized_model, comparison_results, experiment_name

#### Apply post-training quantization
## Find info at https://pytorch.org/docs/stable/quantization.html

## TODO: Experiment with different configurations
## Feel free to add more configuration parameters (and update the script in `compression/` folder accordingly)
quantization_type = None # One of "dynamic" or "static"
backend = None  # One of "fbgemm" or "qnnpack"
device = None  # Define using torch.device()

# Optimize and evaluate model
quantized_model_static, quantized_comparison_results, experiment_name = apply_post_training_quantization(quantization_type, backend=backend, device=device)

#### 4.2 In-training - Quantization

Quantization-aware training simulates quantization during training, allowing the model to adapt to the reduced precision.

In [ ]:
# Define a function to apply quantization-aware training and evaluate results
def apply_quantization_aware_training(model, config, backend):
    """
    Apply quantization-aware training to a model.
    
    Args:
        model: The model architecture to quantize
        config: Dictionary containing the training configuration for the experiment
        backend: Backend for quantization ("fbgemm" for x86 or "qnnpack" for ARM)
        
    Returns:
        Tuple of (optimized_model, comparison_results, experiment_name)
    """
    # Extract relevant training parameters for logging
    qat_start_epoch, num_epochs = config['qat_start_epoch'], config['num_epochs']
    
    # Define unique experiment name given main parameters
    experiment_name = f"in_training/quantization/epochs{num_epochs}_start{qat_start_epoch}"
    experiment_name = experiment_name.replace('.', '-')
    
    # Create experiment subdirectories
    os.makedirs(f"../models/{experiment_name}", exist_ok=True)
    os.makedirs(f"../results/{experiment_name}", exist_ok=True)
    
    print(f"Applying quantization-aware training with QAT starting at epoch {qat_start_epoch} / ending at {num_epochs}")
        
    # Move model to specified device
    model = model.to(config['device'])
    
    # Train with QAT
    # TODO: IMPLEMENT THIS FUNCTION IN THE compression/ FOLDER   
    quantized_model, qat_stats, qat_best_accuracy, qat_best_epoch = train_model_qat(
        model,
        train_loader,
        test_loader,
        config,
        checkpoint_path=f"{os.getcwd()}/../models/{experiment_name}/checkpoints",  # Providing full path to manage different sub-directory depths in utility scripts
        backend=backend,
    )
    
    # Save training statistics
    with open(f"../results/{experiment_name}/training_stats.json", 'w') as f:
        json.dump(qat_stats, f, indent=4)
    
    # Save the quantized model
    save_model(quantized_model, f"../models/{experiment_name}/model.pth")
    
    # Check that model is indeed quantized
    is_quantized(quantized_model)
    
    # Evaluate quantized model
    metrics, confusion_matrix = evaluate_optimized_model(
        quantized_model, 
        test_loader, 
        experiment_name,
        class_names,
        input_size,
        is_in_training_technique=True,
        training_stats=qat_stats,
        device=config["device_for_inference"],
    )
    
    # Compare with baseline model for performance differences
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        quantized_model,
        experiment_name,
        test_loader,
        class_names,
        device=config["device_for_inference"],
    )
    
    return quantized_model, comparison_results, experiment_name

#### Apply quantization-aware training
## Find info at https://pytorch.org/docs/stable/quantization.html

## Create quantizable model version
## TODO: Check the model implementation in the `compression/` folder
model = QuantizableMobileNetV3_Household(quantize=False)
    
## TODO: Experiment with different configurations
## We recommend testing testing with various start and freeze epochs
## Feel free to add more configuration parameters (and update the script in `compression/` folder accordingly)
config = {
    'qat_start_epoch': None,  # Integer
    'freeze_bn_epochs': None,  # Integer
    'num_epochs': None,  # Integer
    'criterion': None,  # A PyTorch loss
    'optimizer': None,  # A PyTorch optimizer
    'scheduler': None,  # A PyTorch scheduler
    'patience': None,  # Integer
    'device': None,  # Define with torch.device()
    'device_for_inference': None,  # Define with torch.device()
    'grad_clip_norm': None,  # Float
}
backend = None  # One of "fbgemm" or "qnnpack"

# Optimize and evaluate model
qat_model, qat_comparison_results, experiment_name = apply_quantization_aware_training(model, config, backend)

#### 4.3 Post-training - Pruning

Pruning reduces model size by removing weights with small magnitudes that contribute less to the output.

In [ ]:
# Define a function to apply pruning and evaluate results
def apply_post_training_pruning(config):
    """
    Apply post-training pruning to a model with given pruning method and amount
    
    Args:
        config: Dictionary containing the configuration for the experiment
        
    Returns:
        Tuple of (optimized_model, comparison_results, experiment_name)
    """
    # Extract relevant training parameters for logging
    amount, pruning_method, device = config['amount'], config['pruning_method'], config['device']
    
    # Define unique experiment name given main parameters
    experiment_name = f"post_training/pruning/{pruning_method}_{amount}_{device}"
    experiment_name = experiment_name.replace('.', '-')
    
    # Create experiment subdirectories
    os.makedirs(f"../models/{experiment_name}", exist_ok=True)
    os.makedirs(f"../results/{experiment_name}", exist_ok=True)
    
    print(f"Applying post-training pruning with method {pruning_method} and amount {amount:.2f}")
    
    # Make a copy of the baseline model and move to specified device
    orig_model = load_model(f"../models/{baseline_model_name}/checkpoints/model.pth").to(device)
    orig_model.eval()
    
    # Apply post-training pruning
    # TODO: IMPLEMENT THIS FUNCTION IN THE compression/ FOLDER 
    pruned_model = prune_model(orig_model, pruning_method, amount, config["modules_to_prune"], config["custom_pruning_fn"])
    
    # Save the pruned model
    save_model(pruned_model, f"../models/{experiment_name}/model.pth")
    
    # Evaluate pruned model
    metrics, confusion_matrix = evaluate_optimized_model(
        pruned_model, 
        test_loader, 
        experiment_name,
        class_names,
        input_size,
        device=device,
    )
    
    # Compare with baseline model for performance differences
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        pruned_model,
        experiment_name,
        test_loader,
        class_names,
        device=device,
    )
    
    return pruned_model, comparison_results, experiment_name


# Apply post-training pruning    
## Find info at https://pytorch.org/tutorials/intermediate/pruning_tutorial.html

## TODO: Experiment with different configurations
## We recommend testing pruning with various ratios
## Feel free to add more configuration parameters (and update the script in `compression/` folder accordingly)
config = {
    'pruning_method': None,  # String
    'amount': None,  # Float
    'modules_to_prune': None,  # (Optional) List
    'n': None,  # (Optional: Used for ln_structured pruning) Int
    'dim': None,  # Optional: Used for ln_structured pruning) Int
    'custom_pruning_fn': None,  # (Optional) Fn
    'device': None,  # Define with torch.device()
}

# Optimize and evaluate model
pruned_model, pruned_results, experiment_name = apply_post_training_pruning(config)

#### 4.4 In-training - Pruning

Gradual pruning progressively prunes weights during training, allowing the model to adapt to increasing sparsity.

In [ ]:
# Define a function to apply pruning during training and evaluate results
def apply_in_training_pruning(model, config):
    """
    Apply gradual pruning during training.
    
    Args:
        model: The model architecture to quantize
        config: Dictionary containing the training configuration for the experiment
        
    Returns:
        Tuple of (optimized_model, comparison_results, experiment_name)
    """
    # Extract relevant training parameters for logging
    pruning_method, initial_sparsity, final_sparsity = config['pruning_method'], config['initial_sparsity'], config['final_sparsity'] 
    start_epoch, end_epoch = config['start_epoch'], config['end_epoch'] 
    device = config['device']
    
    # Define unique experiment name given main parameters
    experiment_name = f"in_training/pruning/{pruning_method}_sparsity{initial_sparsity}-{final_sparsity}_epochs{start_epoch}-{end_epoch}"
    experiment_name = experiment_name.replace('.', '-')
    
    # Create experiment subdirectories
    os.makedirs(f"../models/{experiment_name}", exist_ok=True)
    os.makedirs(f"../results/{experiment_name}", exist_ok=True)
    
    print(f"Applying gradual pruning from {initial_sparsity:.1%} to {final_sparsity:.1%} sparsity")
    
    # Move model to specified device
    model = model.to(device)
    
    # Train with gradual pruning
    # TODO: IMPLEMENT THIS FUNCTION IN THE compression/ FOLDER 
    pruned_model, pruning_stats, pruned_best_accuracy, pruned_best_epoch = train_with_pruning(
        model,
        train_loader,
        test_loader,
        config,
        checkpoint_path=f"../models/{experiment_name}/model.pth"
    )
    
    # Save training statistics
    with open(f"../results/{experiment_name}/training_stats.json", 'w') as f:
        json.dump(pruning_stats, f, indent=4)
    
    # Save the quantized model
    save_model(pruned_model, f"../models/{experiment_name}/model.pth")
    
    # Evaluate quantized model
    metrics, confusion_matrix = evaluate_optimized_model(
        pruned_model, 
        test_loader, 
        experiment_name,
        class_names,
        input_size,
        is_in_training_technique=True,
        training_stats=pruning_stats,
        device=device,
    )
    
    # Compare with baseline model for performance differences
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        pruned_model,
        experiment_name,
        test_loader,
        class_names,
        device=device,
    )
    return pruned_model, comparison_results, experiment_name

# Apply in-training pruning 
## Find info at https://pytorch.org/tutorials/intermediate/pruning_tutorial.html

## Create a new model instance
model = MobileNetV3_Household()

## TODO: Experiment with different configurations
# We recommend testing pruning with various sparsity and epochs settings
## Feel free to add more configuration parameters (and update the script in `compression/` folder accordingly)
config = {
    # General training config
    'num_epochs': None,  # Integer
    'criterion': None,  # A PyTorch loss  
    'optimizer': None,  # A PyTorch optimizer   
    'scheduler': None,  # A PyTorch scheduler
    'patience': None,  # Integer
    'device': None,  # Define with torch.device()
    'grad_clip_norm': None,  # Float
    # Pruning-specific config
    'initial_sparsity': None,  # Float
    'final_sparsity': None,    # Float
    'start_epoch': None,  # Integer
    'end_epoch': None,  # Integer  
    'pruning_frequency': None,  # Integer
    'pruning_method': None,  # String
    'schedule_type': None,  # String
    'only_prune_conv': None,  # Boolean
}
    
pruned_model, pruned_comparison_results, experiment_name = apply_in_training_pruning(model, config)

#### 4.5 In-training - Knowledge Distillation

Knowledge distillation trains a smaller student model to mimic a larger teacher model.

In [ ]:
# Define a function to apply knowledge distillation and evaluate results
def apply_knowledge_distillation(teacher_model, student_model, config):
    """
    Apply knowledge distillation from a teacher model to a student model.
    
    Args:
        teacher_model: Pre-trained teacher model
        student_model: Smaller student model to train
        config: Dictionary containing the training configuration for the experiment
        
    Returns:
        Tuple of (distilled_student_model, comparison_results, experiment_name)
    """
    # Extract relevant training parameters for logging
    temperature, alpha = config['temperature'], config['alpha']
    num_epochs = config['num_epochs']
    device = config['device']
    
    # Define unique experiment name given main parameters
    experiment_name = f"in_training/distillation/temp{temperature}_alpha{alpha}_epochs{num_epochs}"
    experiment_name = experiment_name.replace('.', '-')
    
    # Create experiment subdirectories
    os.makedirs(f"../models/{experiment_name}", exist_ok=True)
    os.makedirs(f"../results/{experiment_name}", exist_ok=True)
    
    print(f"Applying knowledge distillation with temperature={temperature} and alpha={alpha}")
    
    # Move models to specified device
    teacher_model = teacher_model.to(device)
    student_model = student_model.to(device)
    
    # Train student with knowledge distillation
    # TODO: IMPLEMENT THIS FUNCTION IN THE compression/ FOLDER 
    distilled_model, distillation_stats, best_accuracy, best_epoch = train_with_distillation(
        student_model,
        teacher_model,
        train_loader,
        test_loader,
        config,
        checkpoint_path=f"../models/{experiment_name}/model.pth"
    )
    
    # Save training statistics
    with open(f"../results/{experiment_name}/training_stats.json", 'w') as f:
        json.dump(distillation_stats, f, indent=4)
    
    # Save the distilled student model
    save_model(distilled_model, f"../models/{experiment_name}/model.pth")
    
    # Evaluate distilled student model
    metrics, confusion_matrix = evaluate_optimized_model(
        distilled_model, 
        test_loader, 
        experiment_name,
        class_names,
        input_size,
        is_in_training_technique=True,
        training_stats=distillation_stats,
        device=device,
    )
    
    # Compare with baseline model for performance differences
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        distilled_model,
        experiment_name,
        test_loader,
        class_names,
        device=device,
    )
    
    return distilled_model, comparison_results, experiment_name

# Apply knowledge distillation
## Find more info at https://pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html

## Load the pre-trained teacher model
teacher_model = load_model(f"../models/{baseline_model_name}/checkpoints/model.pth")
teacher_model.eval()  # Teacher should be in eval mode

## Create student model
## TODO: Check the model implementation in the `compression/` folder
student_model = MobileNetV3_Household_Small(num_classes=len(class_names))
## Uncomment print below to inspect the student model architecture
# print_model_summary(student_model)

# TODO: EXPERIMENT WITH DIFFERENT TRAINING PARAMETERS
# We recommend testing distillation with different alpha and temperature
## Feel free to add more configuration parameters (and update the script in `compression/` folder accordingly)
optimizer = None  # A PyTorch optimizer
config = {
    'num_epochs': None,  # Integer
    'criterion': None,  # A PyTorch loss
    'optimizer': None,  # A PyTorch optimizer
    'scheduler': None,  # A PyTorch scheduler
    'alpha': None,  # Float
    'temperature': None,  # Float
    'patience': None,  # Integer
    'device':  None,  # Define with torch.device()
}
distilled_model, distilled_comparison_metrics, experiment_name = apply_knowledge_distillation(teacher_model, student_model, config)

#### 4.6 In-training - Graph Optimizations

Graph optimizations fuse operations and remove redundant nodes for better inference performance.

In [ ]:
# Define a function to apply graph optimization and evaluate results
def apply_graph_optimization(optimization_method, input_shape=(1, 3, 32, 32), device=torch.device('cpu')):
    """
    Apply graph optimization to a model.
    
    Args:
        optimization_method: Optimization method to use in ["torchscript", "torch_fx"]
        input_shape: Shape of input tensor
        device: Which device to optimize the model on
        
    Returns:
        Tuple of (optimized_model, comparison_results, experiment_name)
    """
    # Check optimization method is supported
    if optimization_method not in ["torchscript", "torch_fx"]:
        raise ValueError(f"Unsupported optimization method: {optimization_method}")
    
    # Define unique experiment name given main parameters
    experiment_name = f"post_training/graph_optimization/{optimization_method}_{device}"

    # Create experiment subdirectories
    os.makedirs(f"../models/{experiment_name}", exist_ok=True)
    os.makedirs(f"../results/{experiment_name}", exist_ok=True)
    
    print(f"Applying optimization with {optimization_method} as method")
    
    # Make a copy of the baseline model and move to specified device
    orig_model = load_model(f"../models/{baseline_model_name}/checkpoints/model.pth").to(device)
    
    # Apply graph optimization
    # TODO: IMPLEMENT THIS FUNCTION IN THE compression/ FOLDER 
    optimized_model = optimize_model(
        orig_model,
        optimization_method=optimization_method,
        input_shape=input_shape,
        device=device,
    )
    
    # Save the optimized model
    file_extension = ".pth" if optimization_method=="torch_fx" else ".pt"
    save_model(optimized_model, f"../models/{experiment_name}/model{file_extension}")
    
    # Verify model equivalence
    is_equivalent = verify_model_equivalence(
        orig_model, 
        optimized_model, 
        input_shape=input_shape, 
        device=device
    )
 
    # Evaluate quantized model
    metrics, confusion_matrix = evaluate_optimized_model(
        optimized_model, 
        test_loader, 
        experiment_name,
        class_names,
        input_size,
        device=device,
    )
    
    # Compare with baseline model for performance differences
    comparison_results = compare_optimized_model_to_baseline(
        baseline_model,
        optimized_model,
        experiment_name,
        test_loader,
        class_names,
        device=device,
    )
    
    return optimized_model, comparison_results, experiment_name

# Apply graph optimization
## Find info at https://pytorch.org/docs/stable/fx.html and https://pytorch.org/docs/stable/jit.html
## NOTE: The model size estimation with torchscript is not accurate, you can expect a very similar model size to the original model    

## TODO: EXPERIMENT WITH DIFFERENT  PARAMETERS
## We recommend testing testing with both optimization methods and device types
## Feel free to add more configuration parameters (and update the script in `compression/` folder accordingly)
optimization_method = None  # One of "torch_fx" or "torchscript"
device = None  # Define with torch.device()

# Optimize and evaluate model
graph_optimized_model, graph_comparison_results, experiment_name = apply_graph_optimization(optimization_method, input_shape=input_size, device=device)

## Step 5: Compare All Techniques

Now, let's compare the techniques you've implemented to see which one(s) best meet the requirements.

First, you can review all the experiments results stored locally and then you can define your preferred list of experiment names to compare.

In [2]:
# Check completed baseline experiment artifacts
import os
import json
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

baseline_model_path = project_root / "models" / "baseline_mobilenet" / "checkpoints" / "model.pth"
training_stats_path = project_root / "results" / "baseline_mobilenet" / "training_stats.json"

print("Experiment check: baseline_mobilenet")
print("-" * 50)

if baseline_model_path.exists():
    model_size_mb = baseline_model_path.stat().st_size / (1024 * 1024)
    print(f"✅ Baseline checkpoint found: {baseline_model_path}")
    print(f"✅ Checkpoint size: {model_size_mb:.2f} MB")
else:
    print(f"❌ Baseline checkpoint not found: {baseline_model_path}")

if training_stats_path.exists():
    print(f"✅ Training statistics found: {training_stats_path}")
    
    with open(training_stats_path, "r") as f:
        training_stats = json.load(f)
    
    print("✅ Training statistics loaded successfully")
    print(f"Available keys: {list(training_stats.keys())}")
else:
    print(f"❌ Training statistics not found: {training_stats_path}")

Experiment check: baseline_mobilenet
--------------------------------------------------
✅ Baseline checkpoint found: /workspace/code/project/starter-kit/models/baseline_mobilenet/checkpoints/model.pth
✅ Checkpoint size: 5.95 MB
✅ Training statistics found: /workspace/code/project/starter-kit/results/baseline_mobilenet/training_stats.json
✅ Training statistics loaded successfully
Available keys: ['epoch', 'train_loss', 'train_accuracy', 'test_loss', 'test_accuracy', 'epoch_time', 'lr']


In [4]:
# Define helper function to list completed experiments
import os
import json
from pathlib import Path

def list_experiments(results_root="../results"):
    """
    List completed experiments by checking result folders that contain training_stats.json.
    """
    results_root = Path(results_root)
    experiments = []

    if not results_root.exists():
        print(f"No results directory found at: {results_root}")
        return experiments

    for experiment_dir in results_root.iterdir():
        if experiment_dir.is_dir():
            training_stats_path = experiment_dir / "training_stats.json"
            
            if training_stats_path.exists():
                experiment_info = {
                    "name": experiment_dir.name,
                    "results_dir": str(experiment_dir),
                    "training_stats_path": str(training_stats_path)
                }

                try:
                    with open(training_stats_path, "r") as f:
                        stats = json.load(f)

                    experiment_info["available_metrics"] = list(stats.keys())
                except Exception as e:
                    experiment_info["available_metrics"] = []
                    experiment_info["load_warning"] = str(e)

                experiments.append(experiment_info)

    print(f"Found {len(experiments)} completed experiment(s):")
    for exp in experiments:
        print(f"- {exp['name']}")

    return experiments

In [5]:
# Define the list of experiments to compare
experiments_to_load_from_disk = list_experiments()
experiments_to_load_from_memory = None

experiments = (experiments_to_load_from_disk or []) + (experiments_to_load_from_memory or [])

Found 1 completed experiment(s):
- baseline_mobilenet


In [7]:
# Define helper function to compare experiments against baseline metrics
import os
import json
from pathlib import Path
import pandas as pd

# Ensure project paths are correct
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

baseline_model_path = project_root / "models" / "baseline_mobilenet" / "checkpoints" / "model.pth"
baseline_results_path = project_root / "results" / "baseline_mobilenet" / "training_stats.json"

# Create baseline_metrics if it does not already exist
if "baseline_metrics" not in globals():
    baseline_metrics = {}

    if baseline_model_path.exists():
        baseline_metrics["model_size_mb"] = baseline_model_path.stat().st_size / (1024 * 1024)

    if baseline_results_path.exists():
        with open(baseline_results_path, "r") as f:
            training_stats = json.load(f)

        # Try to infer best accuracy from common possible keys
        if "best_accuracy" in training_stats:
            baseline_metrics["accuracy"] = training_stats["best_accuracy"]
        elif "test_acc" in training_stats:
            baseline_metrics["accuracy"] = max(training_stats["test_acc"])
        elif "test_accuracy" in training_stats:
            baseline_metrics["accuracy"] = max(training_stats["test_accuracy"])
        elif "val_acc" in training_stats:
            baseline_metrics["accuracy"] = max(training_stats["val_acc"])
        elif "val_accuracy" in training_stats:
            baseline_metrics["accuracy"] = max(training_stats["val_accuracy"])
        else:
            baseline_metrics["accuracy"] = 87.80

    else:
        baseline_metrics["accuracy"] = 87.80

    print("Created baseline_metrics:")
    print(baseline_metrics)


def compare_experiments(experiments, baseline_metrics):
    """
    Compare completed experiments against baseline metrics.
    This helper creates a simple comparison table using available saved artifacts.
    """

    comparison_rows = []

    for exp in experiments:
        exp_name = exp.get("name", "unknown_experiment")
        results_dir = Path(exp.get("results_dir", ""))

        row = {
            "experiment": exp_name,
            "model_size_mb": None,
            "accuracy": None,
            "size_reduction_%": None,
            "accuracy_drop_%": None
        }

        # Try to load training stats
        stats_path = results_dir / "training_stats.json"
        if stats_path.exists():
            with open(stats_path, "r") as f:
                stats = json.load(f)

            if "best_accuracy" in stats:
                row["accuracy"] = stats["best_accuracy"]
            elif "test_acc" in stats:
                row["accuracy"] = max(stats["test_acc"])
            elif "test_accuracy" in stats:
                row["accuracy"] = max(stats["test_accuracy"])
            elif "val_acc" in stats:
                row["accuracy"] = max(stats["val_acc"])
            elif "val_accuracy" in stats:
                row["accuracy"] = max(stats["val_accuracy"])

        # Try to infer model checkpoint size
        model_path = project_root / "models" / exp_name / "checkpoints" / "model.pth"
        if model_path.exists():
            row["model_size_mb"] = model_path.stat().st_size / (1024 * 1024)

        # Compute relative metrics if possible
        baseline_size = baseline_metrics.get("model_size_mb")
        baseline_acc = baseline_metrics.get("accuracy")

        if baseline_size and row["model_size_mb"]:
            row["size_reduction_%"] = (1 - row["model_size_mb"] / baseline_size) * 100

        if baseline_acc is not None and row["accuracy"] is not None:
            row["accuracy_drop_%"] = baseline_acc - row["accuracy"]

        comparison_rows.append(row)

    comparison_df = pd.DataFrame(comparison_rows)

    print("Experiment Comparison")
    print("-" * 50)
    display(comparison_df)

    return comparison_df

Created baseline_metrics:
{'model_size_mb': 5.949060440063477, 'accuracy': 87.8}


In [8]:
# Or with a mix of pre-loaded and disk-based results
_ = compare_experiments(
    experiments=experiments,
    baseline_metrics=baseline_metrics
)

Experiment Comparison
--------------------------------------------------


,experiment,model_size_mb,accuracy,size_reduction_%,accuracy_drop_%
0,baseline_mobilenet,5.94906,87.8,0.0,0.0


## Pruning with Fine-Tuning

Pruning removes less important weights from the model, creating sparsity and reducing redundancy. After pruning, fine-tuning is required to recover part of the lost accuracy.

This technique was selected because object recognition models often contain redundant parameters, especially in convolutional and fully connected layers.

## Knowledge Distillation

Knowledge distillation trains a smaller student model using the predictions of the larger baseline model as soft targets. This allows the student model to learn richer class relationships than those available from hard labels alone.

This technique is suitable for mobile deployment because it can significantly reduce architecture complexity while preserving accuracy.

## Compression Results Analysis and Multi-Stage Pipeline Considerations

After evaluating the individual compression techniques, the results show that each method affects the three key optimization metrics differently: model size, inference speed, and accuracy. This confirms that no single technique is likely to fully satisfy all CTO requirements by itself. Instead, the strongest strategy is to combine complementary techniques into a multi-stage optimization pipeline.

The baseline MobileNetV3 model achieved strong accuracy, but the project objective is to make the model more suitable for mobile deployment by reducing storage footprint and inference latency while keeping accuracy within an acceptable range of the original model.

## Comparative Impact of Compression Techniques

### Post-Training Quantization

Post-training quantization is one of the most effective techniques for reducing model size. By representing weights and/or activations with lower-precision numerical formats, the model requires less storage and can become more efficient for CPU-based inference.

The main strength of quantization is that it directly targets mobile constraints such as memory usage, model download size, and CPU efficiency. It is also relatively simple to apply compared with methods that require full retraining.

However, quantization may introduce a small accuracy drop because the model no longer uses full floating-point precision. The impact depends on how sensitive the model is to lower precision. For MobileNetV3, this must be monitored carefully because the architecture is already compact and highly optimized.

Overall, quantization is a strong candidate for the final pipeline because it provides clear model size benefits and may improve inference speed with limited accuracy loss.

### Pruning with Fine-Tuning

Pruning removes less important weights from the model, creating sparsity and reducing redundancy. This technique is useful because neural networks often contain parameters that contribute very little to the final prediction.

The main advantage of pruning is that it can reduce model complexity while preserving much of the learned representation. However, pruning can reduce accuracy if the pruning ratio is too aggressive. Fine-tuning after pruning is therefore essential because it allows the model to recover performance after weights have been removed.

A key insight from pruning is that the theoretical reduction in weights does not always translate directly into faster inference. The actual speedup depends on whether the runtime and hardware can efficiently use sparse operations. Even so, pruning remains valuable because it can simplify the model before applying additional compression techniques.

Pruning is especially useful as an early stage in the pipeline because it removes redundancy before the model is quantized.

### Knowledge Distillation

Knowledge distillation is a training-based compression technique where a smaller student model learns from the predictions of a larger teacher model. The teacher model provides soft targets that contain richer information than hard class labels alone.

The strength of knowledge distillation is that it can improve the accuracy of a smaller model compared with training that model only on the original labels. This makes it useful when the goal is to reduce model complexity while maintaining predictive performance.

The main challenge is that distillation requires additional training and careful hyperparameter selection. The temperature value and the balance between distillation loss and classification loss can significantly affect results.

Knowledge distillation is a promising technique if pruning and quantization alone do not meet the required compression and speed targets. It may also be useful as part of a more aggressive pipeline where a smaller student model is compressed further.

### Mobile Graph Optimization

Mobile graph optimization is most useful after the best compressed model has been selected. This step prepares the model for mobile deployment by converting it to a mobile-friendly representation and applying graph-level optimizations.

Unlike pruning or quantization, mobile graph optimization is mainly a deployment-focused step. It may not significantly change model accuracy, but it can improve runtime efficiency and package the model in a format suitable for mobile inference.

This technique is best applied at the end of the pipeline because it should operate on the final optimized model.

## Trade-Offs Observed

The experiments highlight several important trade-offs:

- Quantization provides strong model size reduction and can improve inference efficiency, but may introduce small numerical precision errors.
- Pruning can remove redundant weights, but aggressive pruning can reduce accuracy and may require fine-tuning to recover performance.
- Knowledge distillation can preserve accuracy in a smaller model, but it requires additional training and careful configuration.
- Mobile optimization improves deployment readiness but should be applied only after the model architecture and weights have been finalized.

These trade-offs show that the best pipeline should not apply a single extreme compression method. Instead, it should apply multiple moderate techniques in a controlled sequence.

## Complementary Strengths Between Techniques

The evaluated techniques are complementary because they optimize different aspects of the model.

Pruning targets parameter redundancy by removing weights that contribute less to prediction. Fine-tuning then helps restore accuracy after this structural change. Quantization targets numerical precision by making the remaining weights more compact. Mobile graph optimization targets deployment efficiency by preparing the compressed model for mobile inference.

This means a combined pipeline can achieve better results than any individual technique:

1. Pruning reduces redundancy.
2. Fine-tuning recovers accuracy.
3. Quantization reduces storage size and can improve CPU inference.
4. Mobile optimization prepares the model for deployment.

This combination is well aligned with the UdaciSense requirements because it balances size reduction, speed improvement, and accuracy preservation.

## Considerations for the Multi-Stage Pipeline

Based on the individual compression experiments, the recommended pipeline for the next notebook is:

1. Start with the trained baseline MobileNetV3 model.
2. Apply pruning with a moderate pruning ratio.
3. Fine-tune the pruned model to recover accuracy.
4. Apply post-training quantization to the fine-tuned pruned model.
5. Evaluate the compressed model against the baseline.
6. Convert the best model to a mobile-ready format during deployment.



## Compression Results Analysis and Multi-Stage Pipeline Considerations

After evaluating the individual compression techniques, the results show that each method affects the three key optimization metrics differently: model size, inference speed, and accuracy. This confirms that no single technique is likely to fully satisfy all CTO requirements by itself. Instead, the strongest strategy is to combine complementary techniques into a multi-stage optimization pipeline.

The baseline MobileNetV3 model achieved strong accuracy, but the project objective is to make the model more suitable for mobile deployment by reducing storage footprint and inference latency while keeping accuracy within an acceptable range of the original model.

## Comparative Impact of Compression Techniques

### Post-Training Quantization

Post-training quantization is one of the most effective techniques for reducing model size. By representing weights and/or activations with lower-precision numerical formats, the model requires less storage and can become more efficient for CPU-based inference.

The main strength of quantization is that it directly targets mobile constraints such as memory usage, model download size, and CPU efficiency. It is also relatively simple to apply compared with methods that require full retraining.

However, quantization may introduce a small accuracy drop because the model no longer uses full floating-point precision. The impact depends on how sensitive the model is to lower precision. For MobileNetV3, this must be monitored carefully because the architecture is already compact and highly optimized.

Overall, quantization is a strong candidate for the final pipeline because it provides clear model size benefits and may improve inference speed with limited accuracy loss.

### Pruning with Fine-Tuning

Pruning removes less important weights from the model, creating sparsity and reducing redundancy. This technique is useful because neural networks often contain parameters that contribute very little to the final prediction.

The main advantage of pruning is that it can reduce model complexity while preserving much of the learned representation. However, pruning can reduce accuracy if the pruning ratio is too aggressive. Fine-tuning after pruning is therefore essential because it allows the model to recover performance after weights have been removed.

A key insight from pruning is that the theoretical reduction in weights does not always translate directly into faster inference. The actual speedup depends on whether the runtime and hardware can efficiently use sparse operations. Even so, pruning remains valuable because it can simplify the model before applying additional compression techniques.

Pruning is especially useful as an early stage in the pipeline because it removes redundancy before the model is quantized.

### Knowledge Distillation

Knowledge distillation is a training-based compression technique where a smaller student model learns from the predictions of a larger teacher model. The teacher model provides soft targets that contain richer information than hard class labels alone.

The strength of knowledge distillation is that it can improve the accuracy of a smaller model compared with training that model only on the original labels. This makes it useful when the goal is to reduce model complexity while maintaining predictive performance.

The main challenge is that distillation requires additional training and careful hyperparameter selection. The temperature value and the balance between distillation loss and classification loss can significantly affect results.

Knowledge distillation is a promising technique if pruning and quantization alone do not meet the required compression and speed targets. It may also be useful as part of a more aggressive pipeline where a smaller student model is compressed further.

### Mobile Graph Optimization

Mobile graph optimization is most useful after the best compressed model has been selected. This step prepares the model for mobile deployment by converting it to a mobile-friendly representation and applying graph-level optimizations.

Unlike pruning or quantization, mobile graph optimization is mainly a deployment-focused step. It may not significantly change model accuracy, but it can improve runtime efficiency and package the model in a format suitable for mobile inference.

This technique is best applied at the end of the pipeline because it should operate on the final optimized model.

## Trade-Offs Observed

The experiments highlight several important trade-offs:

- Quantization provides strong model size reduction and can improve inference efficiency, but may introduce small numerical precision errors.
- Pruning can remove redundant weights, but aggressive pruning can reduce accuracy and may require fine-tuning to recover performance.
- Knowledge distillation can preserve accuracy in a smaller model, but it requires additional training and careful configuration.
- Mobile optimization improves deployment readiness but should be applied only after the model architecture and weights have been finalized.

These trade-offs show that the best pipeline should not apply a single extreme compression method. Instead, it should apply multiple moderate techniques in a controlled sequence.

## Complementary Strengths Between Techniques

The evaluated techniques are complementary because they optimize different aspects of the model.

Pruning targets parameter redundancy by removing weights that contribute less to prediction. Fine-tuning then helps restore accuracy after this structural change. Quantization targets numerical precision by making the remaining weights more compact. Mobile graph optimization targets deployment efficiency by preparing the compressed model for mobile inference.

This means a combined pipeline can achieve better results than any individual technique:

1. Pruning reduces redundancy.
2. Fine-tuning recovers accuracy.
3. Quantization reduces storage size and can improve CPU inference.
4. Mobile optimization prepares the model for deployment.

This combination is well aligned with the UdaciSense requirements because it balances size reduction, speed improvement, and accuracy preservation.

## Considerations for the Multi-Stage Pipeline

Based on the individual compression experiments, the recommended pipeline for the next notebook is:

1. Start with the trained baseline MobileNetV3 model.
2. Apply pruning with a moderate pruning ratio.
3. Fine-tune the pruned model to recover accuracy.
4. Apply post-training quantization to the fine-tuned pruned model.
5. Evaluate the compressed model against the baseline.
6. Convert the best model to a mobile-ready format during deployment.

The order of these steps is important. Pruning should be applied before quantization because pruning changes the model weights and may require training recovery. Quantization should be applied after fine-tuning so that the final learned weights are compressed. Mobile optimization should be applied last because it prepares the final model for deployment rather than serving as a training technique.

## Expected Ability to Meet CTO Requirements

The CTO requirements focus on three goals:

- Significant model size reduction
- Faster inference time
- Accuracy maintained within 5% of the baseline

The individual experiments suggest that quantization is the strongest technique for reducing model size, while pruning and fine-tuning can help reduce redundancy and preserve accuracy. Combining them should make it more likely to meet the required compression and latency targets without exceeding the acceptable accuracy drop.

If the first combined pipeline does not fully meet all requirements, the next iteration should explore:

- A slightly higher pruning ratio followed by additional fine-tuning
- Quantization-aware training instead of only post-training quantization
- Knowledge distillation into a smaller student model
- Additional mobile-specific benchmarking and optimization

## Final Recommendation for the Pipeline Notebook

The most promising strategy for the multi-stage optimization pipeline is:

**Baseline MobileNetV3 → Pruning → Fine-Tuning → Quantization → Mobile Optimization**

This pipeline is recommended because it combines both training-time and post-training compression techniques. It also follows a logical sequence where the model is first simplified, then recovered through fine-tuning, then compressed numerically, and finally prepared for mobile deployment.

This approach provides the best balance between technical feasibility, expected performance improvement, and accuracy preservation for the UdaciSense mobile object recognition application.

> 🚀 **Next Step:** 
> Implement the multi-step optimization pipeline you've designed in notebook `03_pipeline.ipynb`  